In [ ]:
from flask import Flask, render_template, request
import pyodbc
import pandas as pd

app = Flask(__name__)

# Fixed: raw strings (r"...") so \SQLEXPRESS doesn't break
conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=DESKTOP-1TM1TG3\SQLEXPRESS;"
    "Database=mohamed_Ramadan_DB;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

def get_data():
    return pd.read_sql_query('SELECT * FROM Sales', conn)

@app.route('/', methods=['GET', 'POST'])
def main():
    if request.method == 'GET':
        df = get_data()

        # Normalize column names: strip whitespace, lowercase, replace spaces with underscores
        df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

        # Convert numeric columns safely
        for col in ['quantity', 'unit_price', 'total_amount']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        # 1. Total Revenue (fixed: 'in df.columns')
        if 'total_amount' in df.columns:
            total_revenue = df['total_amount'].sum()
        else:
            total_revenue = (df['quantity'] * df['unit_price']).sum()
        total_orders = df['order_id'].nunique() if 'order_id' in df.columns else 0
        avg_order_val = (total_revenue / total_orders) if total_orders > 0 else 0
        total_quantity = df['quantity'].sum() if 'quantity' in df.columns else 0
        total_customers = df['customer_id'].nunique() if 'customer_id' in df.columns else 0

        if 'customer_id' in df.columns and 'order_id' in df.columns and total_customers > 0:
            orders_per_customer = df.groupby('customer_id')['order_id'].nunique()
            repeat_customers_count = (orders_per_customer > 1).sum()
            repeat_rate = (repeat_customers_count / total_customers * 100)
        else:
            repeat_rate = 0

        # 7. Top Product
        if 'product_name' in df.columns and 'quantity' in df.columns and not df.empty:
            top_product_series = df.groupby('product_name')['quantity'].sum()
            top_product = top_product_series.idxmax() if not top_product_series.empty else "N/A"
        else:
            top_product = "N/A"

        table_html = df.to_html(classes='dataframe table-auto w-full text-left', index=False)

        return render_template(
            'index.html',
            data=table_html,
            total_revenue=f"${total_revenue:,.2f}",
            total_orders=f"{total_orders:,}",
            avg_order_val=f"${avg_order_val:,.2f}",
            total_quantity=f"{total_quantity:,}",
            total_customers=f"{total_customers:,}",
            repeat_rate=f"{repeat_rate:.1f}%",
            top_product=top_product
        )

if __name__ == '__main__':
    # Enabled debug to display real traceback if any problem persists
    app.run(port=5001)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
C:\Users\mohamed\AppData\Local\Temp\ipykernel_22276\2595087450.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query('SELECT * FROM Sales', conn)
127.0.0.1 - - [23/Sep/2026 02:41:15] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Sep/2026 02:41:16] "GET /favicon.ico HTTP/1.1" 404 -


In [74]:
conn.close()